In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from IPython.display import display

# 1. 데이터 로드 및 기본 전처리
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

df_reviews.columns = df_reviews.columns.str.strip()
df_sample.columns = df_sample.columns.str.strip()
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt', 'total_reviews']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days

# 초기 90일 데이터 필터링
df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()
df_m3['voted_up_numeric'] = df_m3['voted_up'].astype(int)

# ==========================================
feature_df = df_m3.groupby('appid').agg(
    Velocity=('timestamp_created', 'count'),             # 화제성 (리뷰 작성 시간 기록의 개수 = 리뷰 수)
    Sentiment=('voted_up_numeric', 'mean'),              # 민심 (긍정률)
    Loyalty=('author_playtime_at_review', 'mean'),       # 몰입도 (평균 플레이타임)
    Influence=('weighted_vote_score', 'sum')             # 전파력 (도움돼요 총합)
).reset_index()

# Stability (일별 긍정률의 표준편차)
daily_sentiment = df_m3.groupby(['appid', 'days_since_release'])['voted_up_numeric'].mean().reset_index()
stability_df = daily_sentiment.groupby('appid')['voted_up_numeric'].std().reset_index().rename(columns={'voted_up_numeric': 'Stability'})
feature_df = pd.merge(feature_df, stability_df, on='appid', how='left').fillna(0)

# 타겟 변수 결합
target_df = df_sample[['appid', 'total_reviews']].drop_duplicates()
ml_df = pd.merge(feature_df, target_df, on='appid').dropna()

# 3. 머신러닝 (Random Forest) 모델 학습
# X: 5가지 원인 / y: 결과
X = ml_df[['Velocity', 'Sentiment', 'Loyalty', 'Influence', 'Stability']]
y = ml_df['total_reviews']

rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
rf_model.fit(X, y)

# 4. 결과 도출 및 시각화
importance = rf_model.feature_importances_
feature_names = ['화제성 (Velocity)', '민심/평점 (Sentiment)', '몰입도 (Loyalty)', '전파력 (Influence)', '안정도 (Stability)']

importance_df = pd.DataFrame({
    '흥행 지표 (Feature)': feature_names,
    '흥행 기여도 (%)': (importance * 100).round(1)
}).sort_values(by='흥행 기여도 (%)', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print(" [ML 분석 결과] 인디게임 장기 흥행을 결정짓는 핵심 지표 랭킹")
print("="*80)
display(importance_df)
print("="*80)


 [ML 분석 결과] 인디게임 장기 흥행을 결정짓는 핵심 지표 랭킹


,흥행 지표 (Feature),흥행 기여도 (%)
0,전파력 (Influence),58.8
1,몰입도 (Loyalty),30.5
2,안정도 (Stability),6.3
3,민심/평점 (Sentiment),4.3
4,화제성 (Velocity),0.0


In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from IPython.display import display

# 1. 데이터 로드
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

# 컬럼명 공백 제거 및 문자열 변환
df_reviews.columns = df_reviews.columns.str.strip()
df_sample.columns = df_sample.columns.str.strip()
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

if 'price_spy' in df_sample.columns:
    df_sample['price_usd'] = df_sample['price_spy'] * 100

# 날짜 데이터 변환 및 기준일(days_since_release) 계산
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt', 'total_reviews', 'price_usd']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days

# '초기 90일' 데이터만 필터링
df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()
df_m3['voted_up_numeric'] = df_m3['voted_up'].astype(int)

# 2. 게임별 6대 핵심 지표(Features) 생성
feature_df = df_m3.groupby('appid').agg(
    Velocity=('timestamp_created', 'count'),             # 화제성 (리뷰 작성 시간 기록의 개수 = 리뷰 수)
    Sentiment=('voted_up_numeric', 'mean'),              # 민심 (긍정률)
    Loyalty=('author_playtime_at_review', 'mean'),       # 몰입도 (평균 플레이타임)
    Influence=('weighted_vote_score', 'sum')             # 전파력 (도움돼요 총합)
).reset_index()

# Stability (일별 긍정률의 표준편차)
daily_sentiment = df_m3.groupby(['appid', 'days_since_release'])['voted_up_numeric'].mean().reset_index()
stability_df = daily_sentiment.groupby('appid')['voted_up_numeric'].std().reset_index().rename(columns={'voted_up_numeric': 'Stability'})
feature_df = pd.merge(feature_df, stability_df, on='appid', how='left').fillna(0) 

# Value Score (가성비: 플레이타임 / 가격)
price_df = df_sample[['appid', 'price_usd']].drop_duplicates()
feature_df = pd.merge(feature_df, price_df, on='appid')
feature_df['Value_Score'] = feature_df['Loyalty'] / (feature_df['price_usd'].replace(0, 0.01)) # 무료게임 0 나누기 방지

# 타겟 변수(최종 흥행 성적: total_reviews) 결합
target_df = df_sample[['appid', 'total_reviews']].drop_duplicates()
ml_df = pd.merge(feature_df, target_df, on='appid').dropna()

# ==========================================
# 3. 머신러닝 (Random Forest) 모델 학습
# ==========================================

# X: 원인 (6가지 지표) / y: 결과 (최종 대박 규모)
X = ml_df[['Velocity', 'Sentiment', 'Loyalty', 'Influence', 'Stability', 'Value_Score']]
y = ml_df['total_reviews']

rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
rf_model.fit(X, y)

# ==========================================
# 4. 결과 도출 및 시각화 포맷팅
# ==========================================
importance = rf_model.feature_importances_
feature_names = ['화제성 (Velocity)', '민심/평점 (Sentiment)', '몰입도 (Loyalty)', 
                 '전파력 (Influence)', '안정도 (Stability)', '가성비 (Value Score)']

importance_df = pd.DataFrame({
    '흥행 지표 (Feature)': feature_names,
    '흥행 기여도 (%)': (importance * 100).round(1)
}).sort_values(by='흥행 기여도 (%)', ascending=False).reset_index(drop=True)

print(" [ML 분석 결과] 인디게임 장기 흥행을 결정짓는 핵심 지표 랭킹")
display(importance_df)

 [ML 분석 결과] 인디게임 장기 흥행을 결정짓는 핵심 지표 랭킹


,흥행 지표 (Feature),흥행 기여도 (%)
0,전파력 (Influence),55.8
1,몰입도 (Loyalty),22.4
2,가성비 (Value Score),12.0
3,안정도 (Stability),5.7
4,민심/평점 (Sentiment),4.0
5,화제성 (Velocity),0.0
